# Learning digits from "which came before which"

**The data.** An example is a short video: a sequence of 6 handwritten digits (MNIST images), one per frame. For instance, the first training sequence (its images are shown in Section 1) is

> 2 &nbsp; 1 &nbsp; 4 &nbsp; 2 &nbsp; 1 &nbsp; 5

The model never sees these digits as labels: nothing says that frame 0 is a 2, nor even which digits the sequence contains.

**The labels.** Each sequence comes only with the answers to 100 yes/no questions about order. For every ordered pair of digits $(c, c_1)$, from $(0, 0)$ to $(9, 9)$:

> `Before(c, c1)`: is there a $c$ somewhere before a $c_1$?

For the sequence above:

| question | answer | why |
|---|---|---|
| `Before(2, 1)` | yes | the 2 in frame 0 comes before the 1 in frame 1 |
| `Before(1, 2)` | yes | the 1 in frame 1 comes before the 2 in frame 3 |
| `Before(5, 2)` | no | the only 5 is the last frame |
| `Before(2, 2)` | yes | with $c = c_1$ the question means "at least two $c$'s": there are two 2s |
| `Before(4, 4)` | no | there is only one 4 |
| `Before(3, 7)` | no | there is no 3 (and no 7) |

11 of the 100 answers are "yes" for this sequence, about the average at length 6.

**The goal.** From these answers alone, learn a classifier that reads the digit in **one frame**. This is hard because the labels are indirect:
- a "yes" says that a pair of frames in the right order exists, but not which one: `Before(2, 1)` holds for frames (0, 1), (0, 4) and (3, 4);
- a "no" is more informative: `Before(5, 2) = no` says that no frame showing a 5 is followed by a frame showing a 2.

The classifier has to find the one assignment of digits to frames that makes all the answers, over many sequences, true at once.

**The settings.** Training uses **1,000 sequences**, and then only **100**. At test time:
- the same questions are asked on new sequences of 6 frames, and on sequences of **12 frames**, a length never seen in training;
- a **question never trained on** is asked too: `Contains(c)`, "does digit $c$ appear?".

**The comparison.** Both models use the same CNN on each frame, the same training sequences, the same labels and the same training budget:
- **sLTN** knows what `Before` means. One rule ties the labels to the classifier:
  $$\forall (x, y)\; \forall c\; \forall c_1:\; \mathit{Before}(y, c, c_1) \leftrightarrow \exists t, t_1 \mid \mathit{before}(t, t_1):\, \big(\mathit{Is}(\mathit{digit}(x[t]), c) \wedge \mathit{Is}(\mathit{digit}(x[t_1]), c_1)\big)$$
  Here `y` is the sequence's 10 × 10 label matrix, so $\mathit{Before}(y, c, c_1)$ is the given answer to the question $(c, c_1)$. $\mathit{digit}(x[t])$ is the classifier's probabilities for frame $t$, and $\mathit{Is}(p, c)$ the probability that $p$ gives to digit $c$. The rule reads: the answer is yes iff some frame showing $c$ comes before some frame showing $c_1$.
  It only has to learn the digits, and the rule holds at any length.
- **The baseline** has no knowledge: CNN per frame, then a GRU over the sequence, then 100 outputs, trained with binary cross-entropy on the same labels. It has to learn the digits *and* what "before" means, from examples of length 6.

**What this example shows:**
- **Learning without labels on the thing being learned.** The digit classifier gets no frame labels at all, only yes/no answers about whole sequences. One rule ties those answers to the frames, and the classifier learns the digits from it.
- **One rule in place of many labels.** Quantifying over the digit classes turns a single formula into the definition of all 100 questions.
- **Time as a structural dimension.** The frames are indexed by a time dimension, and "frame `t` comes before frame `t1`" is a relation over it, which restricts the pairs of frames the rule looks at (Section 3).
- **The semantics of a quantifier matters.** How "there is a pair of frames" is computed (here, as a maximum) decides whether the rule can be satisfied at all (Section 3).
- **Pairing each sequence with its own labels.** `∀ (x, y)` takes the i-th sequence with the i-th label matrix, instead of every sequence with every matrix.
- **Knowledge in place of data.** The rule says what "before" means, so the network only has to learn the digits; training with 1,000 and with 100 sequences shows how far this goes.
- **Using a model on longer sequences.** The rule doesn't depend on the length: a model trained on 6 frames is used on 12 by rebuilding the time relation for the new length (Section 5).
- **Asking new questions of a learned model.** `Contains(c)` is one more formula over the learned classifier, answered with no new training (Section 6).

This notebook runs one seed on CPU. The same procedure was also run for 5 seeds on a GPU (Section 7).

In [ ]:
import copy
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from torchvision import datasets

import sltn
from sltn import Interpretation, Logic, Signature, kb_backward
from sltn.interpretation.operators.aggregator import AggregMax, AggregPMean

CONFIG = dict(
    T=6,               # frames per training sequence
    T_long=12,         # frames per sequence in the length-generalisation test
    sizes=[1_000, 100],  # training sequences: the main setting, then the low-data one
    n_val=500,
    n_test=2_000,
    batch_size=64,
    steps=3_000,       # gradient steps, the same for both models
    eval_every=250,    # validation, to keep the best checkpoint
    lr=1e-3,
    n_probe=100,       # labelled frames for the linear probe (10 per class)
    seed=0,
)
K = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("sltn:", sltn.__version__)

## 1. Data

- **Frames.** MNIST images scaled to [0, 1]. Training and validation sequences draw their frames from disjoint parts of the MNIST training set (55,000 / 5,000 images). Test sequences draw from the MNIST test set.
- **Labels.** `before[n, a, b] = 1` if some $a$ comes earlier than some $b$ in sequence $n$. `earlier[n, t, a]` marks "an $a$ occurs before position $t$", and `Before(a, b)` holds iff some position $t$ shows a $b$ with an $a$ earlier.
- **True digits.** Used to compute the labels and to score frame accuracy. Neither model sees them.
- **`Contains` labels.** `contains[n, c] = 1` if digit $c$ appears in sequence $n$. Used only at test time.
- **The 100 training sequences are the first 100 of the 1,000**, drawn with the same seed.

In [ ]:
DATA = "../datasets/"
def to_frames(ds):
    return ds.data.float().div(255).unsqueeze(1), ds.targets     # (N, 1, 28, 28), (N,)

train_imgs, train_digits = to_frames(datasets.MNIST(DATA, train=True, download=True))
test_imgs, test_digits = to_frames(datasets.MNIST(DATA, train=False, download=True))
pools = {
    "train": (train_imgs[:55_000], train_digits[:55_000]),
    "val":   (train_imgs[55_000:], train_digits[55_000:]),
    "test":  (test_imgs, test_digits),
}

def contains_labels(d):
    # d: (N, T) true digits -> (N, K)
    return F.one_hot(d, K).amax(dim=1).float()

def before_labels(d):
    # d: (N, T) true digits -> (N, K, K)
    oh = F.one_hot(d, K).float()
    earlier = (oh.cumsum(dim=1) - oh).clamp(max=1)
    return torch.einsum("nta,ntb->nab", earlier, oh).clamp(max=1)

class Sequences:
    # n random sequences of T frames from one pool; frames are looked up per batch
    def __init__(self, pool, n, T, seed):
        imgs, digits = pools[pool]
        g = torch.Generator().manual_seed(seed)
        self.imgs = imgs
        self.idx = torch.randint(0, len(imgs), (n, T), generator=g)
        self.digits = digits[self.idx]                   # hidden: scoring only
        self.before = before_labels(self.digits)
        self.contains = contains_labels(self.digits)     # test only
        self.n, self.T = n, T

    def batch(self, b):
        return self.imgs[self.idx[b]], self.before[b]

    def chunks(self, size=250):
        for s in range(0, self.n, size):
            yield self.batch(torch.arange(s, min(s + size, self.n)))

S = CONFIG["seed"]
trains = {n: Sequences("train", n, CONFIG["T"], seed=1000 + S) for n in CONFIG["sizes"]}
assert torch.equal(trains[100].idx, trains[1_000].idx[:100])   # the 100 are the first 100 of the 1,000
train = trains[1_000]
val = Sequences("val", CONFIG["n_val"], CONFIG["T"], seed=2000 + S)
test = Sequences("test", CONFIG["n_test"], CONFIG["T"], seed=3000 + S)
test_long = Sequences("test", CONFIG["n_test"], CONFIG["T_long"], seed=4000 + S)
for name, ds in [("train", train), ("test", test), ("test, long", test_long)]:
    print(f"{name:10s}: {ds.n:5d} sequences of {ds.T:2d} frames, {ds.before.mean():.0%} of the Before labels "
          f"and {ds.contains.mean():.0%} of the Contains labels are true")

In [ ]:
n = 0
fig, axes = plt.subplots(1, CONFIG["T"], figsize=(1.6 * CONFIG["T"], 1.9))
for t, ax in enumerate(axes):
    ax.imshow(train.imgs[train.idx[n, t], 0], cmap="gray")
    ax.set_title(f"t={t}: {train.digits[n, t].item()}", fontsize=9)
    ax.axis("off")
plt.show()
pairs = [(a, b) for a in range(K) for b in range(K) if train.before[n, a, b]]
print("digits (hidden):", train.digits[n].tolist())
print(f"Before true for {len(pairs)} of 100 pairs:", pairs)

## 2. The networks

Both models use the same CNN on each frame. It has two conv layers and outputs 128 features per frame.
- **sLTN's `digit`:** the CNN, then a linear layer and a softmax over the 10 classes.
- **The baseline:** the CNN on each frame, then a GRU over the 6 (or 12) feature vectors, then a linear layer from its last state to 100 logits, one per `Before(c, c1)`.

In [ ]:
class Trunk(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 28 -> 14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 14 -> 7
            nn.Flatten(), nn.Linear(32 * 7 * 7, 128), nn.ReLU(),
        )

    def forward(self, img):                  # (N, 1, 28, 28) -> (N, 128)
        return self.net(img)

class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk, self.head = Trunk(), nn.Linear(128, K)

    def forward(self, img):                  # (N, 1, 28, 28) -> (N, K) probabilities
        return torch.softmax(self.head(self.trunk(img)), dim=-1)

class GRUBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = Trunk()
        self.gru = nn.GRU(128, 128, batch_first=True)
        self.head = nn.Linear(128, K * K)

    def forward(self, x):                    # (B, T, 1, 28, 28) -> (B, K, K) logits
        B, T = x.shape[:2]
        _, h = self.gru(self.trunk(x.flatten(0, 1)).view(B, T, -1))
        return self.head(h[-1]).view(B, K, K)

## 3. sLTN: the knowledge

**Signature** (the language, with no values yet):
- **`x : Image [T]`.** A sequence: one individual per sequence, with a structural axis over time. `x[t]` is its frame at time `t`.
- **`y`.** The sequence's 10 × 10 label matrix. `∀ (x, y)` goes over sequences and their labels together, not over all pairs.
- **`c`, `c1`.** The 10 classes, as one-hot vectors. `c1` is an alias of `c`.
- **`digit : Image → Probs`.** The classifier. It has no `input_dims`, so it sees one frame at a time: sLTN flattens sequences × time into one batch before calling it.
- **`Is(p, c)`.** The probability that `p` gives to class `c`. It is needed because `digit` is a function, not a predicate: it returns one softmax vector per frame, so the 10 classes compete and each frame gets exactly one digit, and `Is` turns the entry of class `c` into a truth degree. A predicate `Digit(x[t], c)` with a sigmoid per class would let a frame be a 2 and a 7 at once, and would give no single classifier to score in Section 5.
- **`Before(y, c, c1)`.** Reads the label of the pair from `y`.
- **`before(t, t1)`.** The structural relation `t < t1`, used as the guard of `∃ t, t1`.

In [ ]:
sig = Signature("order")
for sort in ("Image", "Class", "Probs", "BeforeLabels"):
    sig.sort(sort)
sig.dimension("T")
sig.variable("x", "Image", dims=["T"])
sig.variable("y", "BeforeLabels")
sig.variable("c", "Class")
sig.structural_variable("t", "T")
sig.function("digit", ["Image"], "Probs")
sig.predicate("Is", ["Probs", "Class"])
sig.predicate("Before", ["BeforeLabels", "Class", "Class"])
sig.structural_relation("before", ["T", "T"])

rule = sig.parse(
    "forall (x, y): forall c: forall c1: (Before(y, c, c1) <-> "
    "exists t, t1 | before(t, t1): (Is(digit(x[t]), c) & Is(digit(x[t1]), c1)))"
)
# The right-hand side alone: the rule's answer to each question, computed from `digit`.
answer = sig.parse("exists t, t1 | before(t, t1): (Is(digit(x[t]), c) & Is(digit(x[t1]), c1))")
# A question never trained on (Section 6): does digit c appear?
contains = sig.parse("exists t: Is(digit(x[t]), c)")
print("rule:    ", rule)
print("answer:  ", answer, "| free variables:", answer.free_variables)
print("contains:", contains, "| free variables:", contains.free_variables)

**Interpretation** (the values):
- `Is` is a dot product, and `Before` looks up `y[c, c1]`.
- `c` is grounded with the 10 one-hot vectors, and `before` with the strict upper triangle of a $T \times T$ matrix.
- The logic is the default one (product ∧, Reichenbach →, which also defines ↔, and pMeanError p = 2 for ∀), plus **max** for ∃ over time.

**Why max for ∃.** The sanity check below gives the rule a perfect classifier: frames replaced by the one-hot vector of their true digit, and `digit` the identity. A correct rule must then reproduce every label.
- With max it does, at both lengths.
- With pMean (p = 2), the aggregator the running example (`0_Running_example`) uses for ∃, one true pair of frames is averaged with all the false pairs. A "yes" answer drops to about 0.26 at length 6 and 0.12 at length 12, so the rule can't be satisfied, and the problem gets worse as sequences get longer.

In [ ]:
def make_interp(digit_fn, image_type, exists="max"):
    logic = Logic().with_defaults()
    logic["exists,t"] = AggregMax() if exists == "max" else AggregPMean(p=exists)
    it = Interpretation(sig, logic=logic)
    it["Image"] = image_type
    it["Class"] = sltn.Type("Class", shape=(K,))
    it["Probs"] = sltn.Type("Probs", shape=(K,))
    it["BeforeLabels"] = sltn.Type("BeforeLabels", shape=(K, K))
    it["digit"] = digit_fn
    it["Is"] = lambda p, c: (p * c).sum(-1, keepdim=True)
    it["Before"] = lambda y, c, c1: torch.einsum("nij,ni,nj->n", y, c, c1).unsqueeze(-1)
    it["c"] = torch.eye(K)
    return it

def ground(it, x, y):
    T = x.shape[1]
    it["x"], it["y"] = x, y
    it["before"] = torch.ones(T, T, device=x.device).triu(diagonal=1)   # before[t, t1] = 1 iff t < t1

def evaluate(it, formula, x, order=("x", "c", "c1")):
    # The value of `formula` on the sequences x, e.g. the rule's answer to the 100 questions
    # as (B, K, K) in [0, 1]. The result's axes are named, so they are reordered by name;
    # the labels don't enter the answer.
    ground(it, x, torch.zeros(x.shape[0], K, K, device=x.device))
    r = it(formula)
    names = [a.name for a in r.axes][:-1]
    return r.value.squeeze(-1).permute(*[names.index(n) for n in order])

for ds in (test, test_long):
    for exists in ("max", 2):
        it = make_interp(lambda v: v, sltn.Type("Image", shape=(K,)), exists)
        x, y = F.one_hot(ds.digits[:200], K).float(), ds.before[:200]
        a = evaluate(it, answer, x)
        ground(it, x, y)
        print(f"T={ds.T:2d}, ∃ = {str(exists):3s}: labels reproduced: {torch.equal((a > 0.5).float(), y)!s:5s} | "
              f"answer on true labels >= {a[y == 1].min():.2f}, on false <= {a[y == 0].max():.2f} | "
              f"rule satisfied {it(rule).value.item():.3f}")

## 4. Training, the same procedure for both

**Two training sizes.** A calibration run before this notebook (one seed, `Before` F1 at length 6) gave:

| training sequences | sLTN | GRU (trained up to 6,000 steps) |
|---|---|---|
| 100 | 0.86 | 0.20 |
| 1,000 | 0.95 | 0.76 |
| 10,000, both label families (earlier prototype) | 0.98 | — |

- **1,000 is the main setting.** The GRU does reasonably well at length 6, so any difference has to come from the knowledge.
- **100 shows the low-data end.** The GRU barely learns there. The question is how much sLTN loses when its data is cut by 10.

- **Budget.** Adam with learning rate 1e-3, 3,000 steps of 64 sequences drawn at random from the training set: about 190 passes over 1,000 sequences, 1,900 over 100. The calibration showed the GRU stops improving by 3,000 steps.
- **Four runs:** each model, on 1,000 and on 100 sequences. Each run starts from the same seed.
- **Validation.** Every 250 steps, both models answer the 100 questions on 500 validation sequences of length 6. The checkpoint with the best validation macro F1 is kept.
- **sLTN.** Loss = 1 − satisfaction of the rule, computed by `kb_backward`. Its answer to a question is the rule's right-hand side, thresholded at 0.5.
- **GRU.** Binary cross-entropy on the 100 labels. Its answer is the sign of the logit.

In [ ]:
def macro_f1(y, pred):
    return f1_score(y.flatten(1).numpy(), pred.flatten(1).numpy(), average="macro", zero_division=0)

def train_model(name, model, loss_step, predict, train):
    torch.manual_seed(S)
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
    g = torch.Generator().manual_seed(S)
    best, curve, t0 = (-1.0, None, 0), [], time.perf_counter()
    for step in range(1, CONFIG["steps"] + 1):
        x, y = train.batch(torch.randint(0, train.n, (CONFIG["batch_size"],), generator=g))
        opt.zero_grad()
        loss = loss_step(opt, x.to(device), y.to(device))
        opt.step()
        if step % CONFIG["eval_every"] == 0:
            f1 = evaluate_f1(predict, val)
            curve.append((step, f1))
            if f1 > best[0]:
                best = (f1, copy.deepcopy(model.state_dict()), step)
            if step % 1000 == 0:
                print(f"{name} step {step:4d}: loss {loss:.4f}, val F1 {f1:.3f} ({time.perf_counter() - t0:.0f}s)")
    model.load_state_dict(best[1])
    print(f"{name}: kept the checkpoint of step {best[2]} (val F1 {best[0]:.3f})")
    return curve

@torch.no_grad()
def evaluate_f1(predict, ds):
    preds = torch.cat([predict(x.to(device)).cpu() for x, _ in ds.chunks()])
    return macro_f1(ds.before, preds)

In [ ]:
kb = {"Before rule": rule}
runs = {}      # (model, n_train) -> the trained network, its interpretation and predict function

def train_sltn(n):
    torch.manual_seed(S)
    net = DigitCNN().to(device)
    it = make_interp(net, sltn.Type("Image", shape=(1, 28, 28))).to(device)

    def loss_step(opt, x, y):
        ground(it, x, y)
        return kb_backward(opt, it, kb)["agg_loss"]

    def predict(x):
        return evaluate(it, answer, x) > 0.5

    curve = train_model(f"sLTN, {n}", net, loss_step, predict, trains[n])
    runs["sLTN", n] = dict(net=net, interp=it, predict=predict, curve=curve)

for n in CONFIG["sizes"]:
    train_sltn(n)

In [ ]:
def train_gru(n):
    torch.manual_seed(S)
    net = GRUBaseline().to(device)

    def loss_step(opt, x, y):
        loss = F.binary_cross_entropy_with_logits(net(x), y)
        loss.backward()
        return loss.item()

    def predict(x):
        return net(x) > 0

    curve = train_model(f"GRU, {n}", net, loss_step, predict, trains[n])
    runs["GRU", n] = dict(net=net, predict=predict, curve=curve)

for n in CONFIG["sizes"]:
    train_gru(n)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for (name, n), r in runs.items():
    ax.plot(*zip(*r["curve"]), marker="o", ms=3, ls="-" if n == 1_000 else "--",
            color="C0" if name == "sLTN" else "C1", label=f"{name}, {n} sequences")
ax.set_xlabel("step"); ax.set_ylabel("val Before macro F1 (T = 6)"); ax.set_ylim(0, 1); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Test

- **`Before` macro F1:** the F1 of each question over the test sequences, then averaged over the 100 questions. Accuracy would mislead, because about 89% of the answers are "no" at length 6. Measured at length 6, and at length 12 with no retraining. For sLTN, the `before` relation is regrounded at size 12×12. The GRU simply runs over more frames.
- **Frame accuracy, sLTN's classifier:** the argmax of `digit` against the true digit. This uses no frame labels.
- **Frame accuracy, linear probe:** the GRU has no per-frame digit output, so its frame features are scored this way: a logistic regression on the 128 CNN features, fitted on 100 labelled validation frames (10 per class), scored on the test frames. It is fitted on every trained CNN and on an untrained one, which gives the floor.
- **`Contains` macro F1:** a question never trained on, in Section 6.

In [ ]:
@torch.no_grad()
def probe_acc(trunk, ds):
    imgs, digits = pools["val"]
    idx = torch.cat([torch.nonzero(digits == k).flatten()[: CONFIG["n_probe"] // K] for k in range(K)])
    clf = LogisticRegression(max_iter=2000).fit(trunk(imgs[idx].to(device)).cpu().numpy(), digits[idx].numpy())
    feats = torch.cat([trunk(x.flatten(0, 1).to(device)).cpu() for x, _ in ds.chunks()]).numpy()
    return (clf.predict(feats) == ds.digits.flatten().numpy()).mean()

@torch.no_grad()
def direct_acc(net, ds):
    preds = torch.cat([net(x.flatten(0, 1).to(device)).argmax(-1).cpu() for x, _ in ds.chunks()])
    return (preds == ds.digits.flatten()).float().mean().item()

for r in runs.values():
    r["net"].eval()
torch.manual_seed(S)
untrained = Trunk().to(device).eval()
rows = {}
for (name, n), r in runs.items():
    rows[f"{name}, {n}"] = dict(
        direct=direct_acc(r["net"], test) if name == "sLTN" else None,
        probe=probe_acc(r["net"].trunk, test),
        f1_6=evaluate_f1(r["predict"], test), f1_12=evaluate_f1(r["predict"], test_long))
rows["untrained CNN (floor)"] = dict(direct=None, probe=probe_acc(untrained, test), f1_6=None, f1_12=None)
fmt = lambda v: "   —  " if v is None else f"{v:.3f} "
print(f"{'':22s} | frame acc, no labels | frame acc, probe (100 frames) | Before F1, T=6 | Before F1, T=12")
for name, r in rows.items():
    print(f"{name:22s} | {fmt(r['direct']):>20s} | {fmt(r['probe']):>29s} | {fmt(r['f1_6']):>14s} | {fmt(r['f1_12']):>15s}")

Where does each model go wrong? Below is the F1 of each question at length 12, for the models trained on 1,000 sequences, as a 10 × 10 grid: row = $c$, column = $c_1$, with the "two $c$'s" questions on the diagonal.

In [ ]:
@torch.no_grad()
def per_question_f1(predict, ds):
    preds = torch.cat([predict(x.to(device)).cpu() for x, _ in ds.chunks()]).flatten(1).numpy()
    return f1_score(ds.before.flatten(1).numpy(), preds, average=None, zero_division=0).reshape(K, K)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
for ax, name in zip(axes, ("sLTN", "GRU")):
    im = ax.imshow(per_question_f1(runs[name, 1_000]["predict"], test_long), vmin=0, vmax=1, cmap="viridis")
    ax.set_title(f"{name}: F1 per question, T = 12"); ax.set_xlabel("c1"); ax.set_ylabel("c")
    ax.set_xticks(range(K)); ax.set_yticks(range(K))
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

## 6. A question never trained on: `Contains(c)`

"Does digit $c$ appear in the sequence?" No model was trained on it. Its positive rate is about 47% at length 6 and 72% at length 12.

- **sLTN states it in the same language and evaluates it on the learned classifier**, with no new training:
  $$\mathit{Contains}(c) := \exists t:\, \mathit{Is}(\mathit{digit}(x[t]), c)$$
- **The GRU has no output for it**, and a new head would need `Contains` labels and training. Its 100 `Before` answers can still be turned into one, if one knows by hand that, for sequences of at least 2 frames, $c$ appears iff it comes before or after some digit:
  $$\mathit{Contains}(c) \iff \exists c_1:\, \mathit{Before}(c, c_1) \vee \mathit{Before}(c_1, c)$$
  This is the GRU's best chance: it uses knowledge the GRU doesn't have. The same derivation is also applied to sLTN's `Before` answers, for comparison.

In [ ]:
def contains_from_before(before_pred):
    # (B, K, K) bool -> (B, K): c appears iff it is before or after some digit (T >= 2)
    return before_pred.any(dim=2) | before_pred.any(dim=1)

@torch.no_grad()
def contains_f1(r, ds, derived):
    if derived:
        pred = torch.cat([contains_from_before(r["predict"](x.to(device))).cpu() for x, _ in ds.chunks()])
    else:
        pred = torch.cat([(evaluate(r["interp"], contains, x.to(device), order=("x", "c")) > 0.5).cpu()
                          for x, _ in ds.chunks()])
    return macro_f1(ds.contains, pred)

print(f"{'Contains, macro F1':38s} | T = 6 | T = 12")
for (name, n), r in runs.items():
    ways = [("the formula", False), ("derived from Before", True)] if name == "sLTN" else [("derived from Before", True)]
    for way, derived in ways:
        print(f"{name + ', ' + str(n) + ', ' + way:38s} | {contains_f1(r, test, derived):.3f} | {contains_f1(r, test_long, derived):.3f}")

## 7. Findings

One seed, on CPU: about 5.2 min per sLTN run and 2.7 min per GRU run.

| | training sequences | frame acc, no frame labels | frame acc, probe (100 frames) | `Before` F1, T = 6 | `Before` F1, T = 12 | `Contains` F1, never trained, T = 6 | T = 12 |
|---|---|---|---|---|---|---|---|
| sLTN | 1,000 | **0.976** | 0.955 | **0.959** | **0.969** | **0.982** | **0.987** |
| GRU baseline | 1,000 | — | 0.955 | 0.772 | 0.268 | 0.959 * | 0.787 * |
| sLTN | 100 | **0.930** | 0.912 | **0.887** | **0.918** | **0.950** | **0.968** |
| GRU baseline | 100 | — | 0.738 | 0.195 | 0.122 | 0.625 * | 0.599 * |
| untrained CNN (floor) | — | — | 0.588 | — | — | — | — |

\* derived from the GRU's `Before` answers with a rule written by hand (Section 6); the GRU has no `Contains` output.

**1. With the rule, the order questions are answered better, and the answers carry over to longer sequences.**
- With 1,000 sequences, at length 6, sLTN reaches 0.96 against the GRU's 0.77.
- At length 12, sLTN stays at 0.97, while the GRU drops to 0.27. The GRU learned an answer that works for sequences of 6 frames. The rule's answer is the same at any length: only the `before` grid is regrounded.

**2. With 10 times less data, sLTN loses a little and the GRU collapses.**
- From 1,000 to 100 sequences, sLTN's `Before` F1 goes from 0.96 to 0.89 at length 6 (0.97 to 0.92 at length 12), and its classifier still names 93% of the frames with no frame labels.
- The GRU goes from 0.77 to 0.20, and to 0.12 at length 12.
- sLTN with 100 sequences beats the GRU with 1,000 at both lengths.

**3. A question never trained on is answered by sLTN as well as the trained ones.**
- `Contains(c)` is one more formula, `∃ t: Is(digit(x[t]), c)`, evaluated on the learned classifier: F1 0.98 / 0.99 with 1,000 sequences, 0.95 / 0.97 with 100.
- The GRU can only answer it through a derivation written by hand outside the model. Even so, it reaches 0.96 at length 6 but 0.79 at length 12, and about 0.6 with 100 sequences.
- For sLTN, the formula and the same derivation give identical scores: both read the same, confident classifier. The difference is that the formula is a question in the model's own language, not code written around it.

**4. With 1,000 sequences, both models learn equally good digit features; with 100, only sLTN does.**
- With 1,000, the linear probe reads digits from the GRU's CNN as well as from sLTN's (0.955 each, against 0.59 for an untrained CNN). So the GRU *does* learn to tell digits apart, and what it lacks is the meaning of "before", which it has to infer from examples, and which doesn't extend to 12 frames.
- With 100, the GRU's features drop to 0.74 and sLTN's stay at 0.91.

**5. sLTN learns much faster.** With 1,000 sequences, its validation F1 is 0.96 at step 1,000; the GRU's is 0.10 there and 0.74 at step 2,000 (see the curves in Section 4).

**6. Satisfaction isn't the score.** sLTN's training loss (1 − satisfaction) falls to about 0.002 with 1,000 sequences, while its F1 stays near 0.96: the rule can be almost satisfied on the training batches without every answer being right on new sequences.

**Limits of this result:**
- **One seed here, but the result holds over 5.** The same procedure was run for 5 seeds of every run above on a GPU. Across seeds, every score varies by at most 0.02 (standard deviation), far less than the gaps between the models, and seed 0 on the GPU is within 0.01 of the numbers above. So the conclusions don't depend on the seed.
- **One baseline.** The GRU is a reasonable no-knowledge model, not the strongest possible. A model trained on mixed lengths, or one with attention, would probably do better at length 12. The comparison measures what *exact* knowledge of the label definition is worth against a model that has to learn it.
- **The rule is exactly how the labels were made.** That is the point of the task, but with noisy or partly wrong knowledge the gap would be smaller.